In [ ]:
import math

import equinox as eqx
import jax
import jax.numpy as jnp
from context_flux_no.models.multiphysics.disco import DISCO
from context_flux_no.models.multiphysics.disco.operatornet import OperatorNetwork
from context_flux_no.utils import num_parameters
from jaxtyping import Array, Int, PRNGKeyArray

In [ ]:
disco = DISCO(
    num_spatial_dims=2,
    channels=3,
    embedding_dim=384,
    patch_size=16,
    num_hypernet_blocks=6,
    droppath=0.1,
    num_hypernet_heads=3,
    mlp_hidden_dim=384,
    boundary_condition="periodic",
    key=jax.random.key(0),
)
disco_jit = eqx.filter_jit(disco)
test_arr = jnp.zeros((10, 3, 64, 64))

In [ ]:
%%timeit
disco_jit(test_arr, key=jax.random.key(1)).block_until_ready()

In [ ]:
%%timeit
disco(test_arr, key=jax.random.key(1)).block_until_ready()

In [ ]:
num_parameters(disco)

In [ ]:
opnet = OperatorNetwork(
    num_spatial_dims=2,
    channels=3,
    hidden_channels_base=8,
    groups_norm=4,
    boundary_condition="periodic",
    key=jax.random.key(0),
)
opnet

In [ ]:
scale = jax.tree.map(
    lambda x: float(jnp.max(jnp.abs(x))),
    eqx.filter(opnet, eqx.is_inexact_array),
)

In [ ]:
def scale_fn(weight: Array, scale: float, factor: float = 2.0):
    scale_ = scale * factor
    return scale_ * (2 * jax.nn.sigmoid(weight / scale_) - 1)


jax.tree.map(scale_fn, eqx.filter(opnet, eqx.is_inexact_array), scale)

In [ ]:
tuple((1, 2, 3))

In [ ]:
head = HyperNetworkHead(
    in_size=20,
    base_network=opnet,
    where=lambda net: (
        *net.input_block.layers[1:],
        *net.output_block.layers[1:],
        net.downsampling_blocks,
        net.upsampling_blocks,
    ),
    key=jax.random.key(0),
)

In [ ]:
head.param_vector_to_target(jnp.zeros((head.out_size,)))

In [ ]:
head(jnp.ones((20,)), opnet)

In [ ]:
num_parameters((opnet.upsampling_blocks, opnet.downsampling_blocks))

In [ ]:
params, static = eqx.partition(opnet, eqx.is_inexact_array)
jax.tree.map(lambda x: x.shape, params)

In [ ]:
jax.flatten_util.ravel_pytree(params.bottleneck)[0].shape

jax.flatten_util.ravel_pytree((params.upsampling_blocks, params.downsampling_blocks))[
    0
].shape

## Naive implementation

A direct JAX/Equinox translation of https://gist.github.com/huchenxucs/c65524185e8e35c4bcfae4059f896c16

In [ ]:
class RelativePositionBias(eqx.Module):
    bidirectional: bool = eqx.field(static=True)
    num_buckets: int = eqx.field(static=True)
    max_distance: int = eqx.field(static=True)
    num_heads: int = eqx.field(static=True)

    embedding: eqx.nn.Embedding

    def __init__(
        self,
        bidirectional: bool = True,
        num_buckets: int = 32,
        max_distance=128,
        num_heads=2,
        *,
        key: PRNGKeyArray,
    ):
        self.bidirectional = bidirectional
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        self.num_heads = num_heads
        self.embedding = eqx.nn.Embedding(self.num_buckets, self.num_heads, key=key)

    def _relative_position_bucket(
        self,
        relative_position: Int[Array, "query_size key_size"],
    ):
        """
        Adapted from Mesh Tensorflow:
        https://github.com/tensorflow/mesh/blob/0cb87fe07da627bf0b7e60475d59f95ed6b5be3d/mesh_tensorflow/transformer/transformer_layers.py#L593
        Translate relative position to a bucket number for relative attention.
        The relative position is defined as memory_position - query_position, i.e.
        the distance in tokens from the attending position to the attended-to
        position.  If bidirectional=False, then positive relative positions are
        invalid.
        We use smaller buckets for small absolute relative_position and larger buckets
        for larger absolute relative_positions.  All relative positions >=max_distance
        map to the same bucket.  All relative positions <=-max_distance map to the
        same bucket.  This should allow for more graceful generalization to longer
        sequences than the model has been trained on.
        Args:
            relative_position: an int32 Tensor
            bidirectional: a boolean - whether the attention is bidirectional
            num_buckets: an integer
            max_distance: an integer
        Returns:
            a Tensor with the same shape as relative_position, containing int32
            values in the range [0, num_buckets)
        """
        ret = 0
        n = -relative_position
        if self.bidirectional:
            num_buckets = self.num_buckets // 2
            ret += (n < 0).astype(
                int
            ) * num_buckets  # mtf.to_int32(mtf.less(n, 0)) * num_buckets
            n = jnp.abs(n)
        else:
            num_buckets = self.num_buckets
            n = jnp.maximum(n, jnp.zeros_like(n))
        # now n is in the range [0, inf)

        # half of the buckets are for exact increments in positions
        max_exact = num_buckets // 2
        is_small = n < max_exact
        # The other half of the buckets are for logarithmically bigger bins in positions up to max_distance
        val_if_large = max_exact + (
            jnp.log(n.astype(float) / max_exact)
            / math.log(self.max_distance / max_exact)
            * (num_buckets - max_exact)
        ).astype(int)
        val_if_large = val_if_large.astype(int)
        val_if_large = jnp.minimum(
            val_if_large, jnp.full_like(val_if_large, num_buckets - 1)
        )

        ret += jnp.where(is_small, n, val_if_large)
        return ret

    def compute_bias(self, qlen, klen):
        """Compute binned relative position bias"""
        context_position = jnp.arange(qlen, dtype=int)[:, None]
        memory_position = jnp.arange(klen, dtype=int)[None, :]
        relative_position = memory_position - context_position  # shape (qlen, klen)
        """
                   k
             0   1   2   3
        q   -1   0   1   2
            -2  -1   0   1
            -3  -2  -1   0
        """
        rp_bucket = self._relative_position_bucket(
            relative_position,  # shape (qlen, klen)
        )
        print(rp_bucket)
        values = jax.vmap(jax.vmap(self.embedding))(
            rp_bucket
        )  # shape (qlen, klen, num_heads)
        values = jnp.expand_dims(jnp.transpose(values, axes=(2, 0, 1)), axis=0)
        return values

    def forward(self, qlen, klen):
        return self.compute_bias(qlen, klen)  # shape (1, num_heads, qlen, klen)

## Alternative implementation

See context_flux_no.nn.position_encoding.relative.T5RelativePositionalEncoding

In [ ]:
from context_flux_no.nn.position_encoding import T5RelativePositionalEncoding

In [ ]:
pos_enc = T5RelativePositionalEncoding(
    num_encodings=16,
    encoding_size=2,
    max_distance=16,
    bidirectional=True,
    key=jax.random.key(0),
)

In [ ]:
pos_enc_naive = RelativePositionBias(
    bidirectional=True,
    num_buckets=16,
    max_distance=16,
    num_heads=2,
    key=jax.random.key(0),
)

In [ ]:
query_size = 32
key_size = 32
relative_position = jnp.arange(key_size, dtype=int) - jnp.arange(
    query_size, dtype=int
).reshape(-1, 1)
relative_position

In [ ]:
bin1 = pos_enc.relative_position_to_embedding_bin(relative_position)

In [ ]:
bin2 = pos_enc_naive._relative_position_bucket(relative_position)

In [ ]:
bin2

In [ ]:
pos_enc.embedding.weight

In [ ]:
pos_enc_naive.embedding.weight

In [ ]:
enc_naive = pos_enc_naive.forward(32, 32)

In [ ]:
enc = pos_enc(32, 32)

In [ ]:
enc_naive

In [ ]:
enc

In [ ]:
from einops import rearrange


jnp.array_equal(enc, rearrange(enc_naive, "1 c q k -> q k c"))